# Fixlane Concern Parser — Fine-Tuning Notebook

This notebook fine-tunes Qwen2.5-3B using QLoRA on the Fixlane concern-parsing dataset.

**Runtime:** Google Colab with a T4 GPU (free tier is fine).

**Expected run time:** ~25-35 minutes end to end.

Just run all cells from top to bottom.

## Step 1 — Install dependencies

In [ ]:
# Install everything we need.
# bitsandbytes is needed for 4-bit quantization (QLoRA).
# peft is the Hugging Face library that gives us LoRA.
!pip install -q transformers peft datasets accelerate bitsandbytes tqdm

## Step 2 — Upload the data files

Upload `train.jsonl`, `val.jsonl`, and `test.jsonl` from the `data/` folder using the cell below, or mount your Google Drive.

In [ ]:
# Option A: Upload files directly from your computer
# This will pop up a file picker in Colab.
from google.colab import files

print("Please upload train.jsonl, val.jsonl, and test.jsonl")
uploaded = files.upload()

# After uploading, the files will be in the current directory (/content/)
import os
for fname in uploaded.keys():
    print(f"Uploaded: {fname}")

# Set paths based on where Colab puts uploaded files
TRAIN_PATH = "/content/train.jsonl"
VAL_PATH   = "/content/val.jsonl"
TEST_PATH  = "/content/test.jsonl"

# Where we'll save the trained adapter weights
OUTPUT_DIR = "/content/adapter_weights"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 3 — Import libraries

In [ ]:
import json
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print("Libraries loaded!")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

## Step 4 — Load and inspect the data

In [ ]:
# Simple helper to read a .jsonl file line by line
def load_jsonl(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip blank lines just in case
                rows.append(json.loads(line))
    return rows

train_data = load_jsonl(TRAIN_PATH)
val_data   = load_jsonl(VAL_PATH)
test_data  = load_jsonl(TEST_PATH)

print(f"Loaded {len(train_data)} train, {len(val_data)} val, {len(test_data)} test examples")
print()
print("Example training row:")
print("  Input:", train_data[0]["input"])
print("  Output:", json.dumps(train_data[0]["output"], indent=4))

## Step 5 — Data quality check and cleaning

The dataset card warned that some rows are noisy or mislabeled. Let's check for a few
specific issues before training.

In [ ]:
# Valid values for the classification fields
VALID_SYSTEMS    = {"engine", "electrical", "brakes", "suspension", "hvac", "drivetrain", "body", "other"}
VALID_SEVERITIES = {"safety_critical", "drivability", "comfort", "cosmetic"}

# --- Check 1: Are there any rows with invalid field values? ---
invalid_rows = []
for i, row in enumerate(train_data):
    out = row["output"]
    if out["vehicle_system"] not in VALID_SYSTEMS:
        invalid_rows.append((i, "bad vehicle_system", row))
    if out["severity"] not in VALID_SEVERITIES:
        invalid_rows.append((i, "bad severity", row))

print(f"Rows with invalid field values: {len(invalid_rows)}")

# --- Check 2: Look for near-duplicate inputs ---
# (The dataset has some inputs repeated with minor wording changes.)
seen_inputs = {}
duplicate_count = 0
for i, row in enumerate(train_data):
    normalized = row["input"].lower().strip()
    if normalized in seen_inputs:
        duplicate_count += 1
    else:
        seen_inputs[normalized] = i

print(f"Exact-duplicate inputs in training set: {duplicate_count}")

# --- Check 3: Spot-check rows that look suspicious ---
# Brakes labeled as 'comfort' is borderline — morning squeak that goes away
# is arguably not safety-critical, so we'll keep these as-is and note
# them as label ambiguity in the report.
print()
print("Suspicious severity labels (brakes marked as comfort):")
for row in train_data:
    if row["output"]["vehicle_system"] == "brakes" and row["output"]["severity"] == "comfort":
        print(f'  "{row["input"]}" -> severity: {row["output"]["severity"]}')

# Decision: We keep all rows as-is because:
# 1. No invalid field values
# 2. The duplicate inputs are slightly paraphrased (different wording)
#    so they could still help the model generalize
# 3. The brakes/comfort rows are genuinely ambiguous, not clearly wrong
print()
print("Decision: keeping all rows. Noise is documented in report.")

## Step 6 — Build the prompt format

We'll format each example as an instruction → response pair.
The model needs to see both the instruction AND the expected JSON output during training
so it learns to generate the right structure.

In [ ]:
# This is the instruction template we use for every example.
# Keeping it simple and consistent is important — if we change the
# wording at inference time, the model might get confused.
INSTRUCTION = """You are a vehicle repair assistant. Parse the technician note below into structured fields.

Respond with a valid JSON object containing exactly these fields:
- vehicle_system: one of [engine, electrical, brakes, suspension, hvac, drivetrain, body, other]
- primary_symptom: a short noun phrase describing the symptom
- severity: one of [safety_critical, drivability, comfort, cosmetic]
- suggested_diagnostic: a short imperative phrase, or null

Only output the JSON object. No extra text."""


def build_prompt(technician_note):
    # This builds the input side of the prompt (no answer included)
    prompt = INSTRUCTION + "\n\nTechnician note: " + technician_note + "\n\nJSON:"
    return prompt


def build_full_training_text(technician_note, output_dict):
    # During training, we give the model the full text: prompt + expected output.
    # The model learns to complete the prompt with the correct JSON.
    prompt = build_prompt(technician_note)
    answer = json.dumps(output_dict)  # convert the output dict to a JSON string
    full_text = prompt + " " + answer
    return full_text, prompt


# Quick sanity check
sample_input = train_data[0]["input"]
sample_output = train_data[0]["output"]
full_text, prompt_only = build_full_training_text(sample_input, sample_output)

print("=== PROMPT ONLY ===")
print(prompt_only)
print()
print("=== FULL TRAINING TEXT (prompt + answer) ===")
print(full_text)

## Step 7 — Load the tokenizer and model (with 4-bit quantization)

We use QLoRA, which loads the base model in 4-bit precision to save GPU memory,
then trains a small set of LoRA adapter layers on top.
This way the full 3B model fits on a T4 GPU.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Load the tokenizer first — this handles converting text to token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Make sure the tokenizer has a padding token set.
# Some models don't set this by default and it causes errors during batching.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# We pad on the right side — this is the standard for causal language models
tokenizer.padding_side = "right"

print("Tokenizer loaded!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

In [ ]:
# Set up 4-bit quantization config (this is the 'Q' in QLoRA)
# 4-bit means the base model weights are stored in 4 bits instead of 16,
# which cuts memory use roughly in half.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # nf4 is the quantization format designed for QLoRA
    bnb_4bit_compute_dtype=torch.float16,  # do math in fp16 even though weights are 4bit
    bnb_4bit_use_double_quant=True,     # double quantization saves a bit more memory
)

# Load the base model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",       # automatically puts the model on the GPU
    trust_remote_code=True,
)

print("Base model loaded in 4-bit!")
print(f"Model parameters: {model.num_parameters():,}")

## Step 8 — Attach LoRA adapters

LoRA adds small trainable weight matrices to the attention layers.
We only update these adapter weights during training — the base model
weights are frozen. This keeps training fast and the checkpoint small.

In [ ]:
# LoRA config — these are the key hyperparameters
#
# r=16: the 'rank' of the LoRA matrices. Higher rank = more capacity
#       but more parameters. r=16 is a reasonable middle ground.
#
# lora_alpha=32: the scaling factor. Usually set to 2x the rank.
#                It controls how much the LoRA weights contribute
#                relative to the frozen base weights.
#
# target_modules: which layers to apply LoRA to.
#                 We target the query, key, value, and output projection
#                 layers in the self-attention blocks because those are
#                 where the model learns how to pay attention to tokens.
#                 For Qwen2.5, these layer names are q_proj, k_proj,
#                 v_proj, and o_proj.
#
# lora_dropout=0.05: a tiny bit of dropout to reduce overfitting.
#
# bias='none': we don't add LoRA to the bias terms, just the weight matrices.

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Wrap the base model with the LoRA adapters
model = get_peft_model(model, lora_config)

# Show how many parameters are actually being trained
# (should be a small fraction of the total)
model.print_trainable_parameters()

## Step 9 — Tokenize the dataset

We need to convert each training example from text into token IDs.
We also set up labels — during training the model tries to predict
the label tokens. We mask out the prompt tokens with -100 so the model
only learns to generate the JSON answer, not repeat the prompt.

In [ ]:
# Max token length — inputs are short (10-20 words) and outputs are
# small JSON objects, so 256 tokens is more than enough.
MAX_LENGTH = 256


def tokenize_example(row):
    """
    Takes one training row and returns input_ids + labels.
    Labels are -100 for the prompt part (so the loss ignores it)
    and the actual token IDs for the answer part.
    """
    technician_note = row["input"]
    output_dict = row["output"]

    # Build the full text and the prompt-only text
    full_text, prompt_only = build_full_training_text(technician_note, output_dict)

    # Tokenize the full text (prompt + answer)
    full_tokenized = tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

    # Tokenize just the prompt so we know where it ends
    prompt_tokenized = tokenizer(
        prompt_only,
        max_length=MAX_LENGTH,
        truncation=True,
    )

    prompt_len = len(prompt_tokenized["input_ids"])

    # Build the labels list:
    # - Set prompt token positions to -100 (the Trainer ignores -100 in loss calculation)
    # - Keep the answer token positions as-is
    input_ids = full_tokenized["input_ids"]
    labels = []
    for i in range(len(input_ids)):
        if i < prompt_len:
            labels.append(-100)  # ignore this token in the loss
        else:
            labels.append(input_ids[i])  # learn to predict this token

    full_tokenized["labels"] = labels
    return full_tokenized


# Tokenize all the splits
print("Tokenizing train set...")
train_tokenized_list = [tokenize_example(row) for row in train_data]

print("Tokenizing val set...")
val_tokenized_list = [tokenize_example(row) for row in val_data]

# Convert to Hugging Face Dataset format (the Trainer expects this)
train_dataset = Dataset.from_list(train_tokenized_list)
val_dataset   = Dataset.from_list(val_tokenized_list)

print(f"\nTrain dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")

## Step 10 — Set up training arguments and run training

In [ ]:
# Training hyperparameters
#
# num_train_epochs=3: run through the training set 3 times.
#                     With only 500 examples this is enough to converge
#                     without overfitting badly.
#
# per_device_train_batch_size=4: process 4 examples at a time.
#                                On a T4 this is safe with 4-bit quantization.
#
# gradient_accumulation_steps=4: accumulate gradients over 4 steps before
#                                 updating weights. This gives us an effective
#                                 batch size of 4 * 4 = 16, which helps stability.
#
# learning_rate=2e-4: a standard starting point for LoRA fine-tuning.
#                     Higher than typical full fine-tuning because LoRA
#                     adapters start from zero and need to move fast.
#
# warmup_steps=10: gradually ramp up the learning rate for the first 10 steps.
#                   Helps avoid large gradient updates at the very start.

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=20,
    evaluation_strategy="epoch",   # evaluate on the val set after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,   # keep the checkpoint with the best val loss
    fp16=True,                     # use fp16 mixed precision to speed things up
    report_to="none",              # don't log to wandb or other services
    optim="paged_adamw_8bit",      # 8-bit Adam saves more memory on T4
)

print("Training arguments set!")

In [ ]:
# The data collator handles padding all the examples in a batch to the same length
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # make sure padding positions get -100 labels
    pad_to_multiple_of=8,     # pad to multiples of 8 for fp16 efficiency
)

# Set up the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Trainer is ready. Starting training...")
print(f"Total training steps: approx {len(train_dataset) // (4 * 4) * 3}")

In [ ]:
# Run the training loop!
trainer.train()

## Step 11 — Save the adapter weights

In [ ]:
# Save only the LoRA adapter weights — NOT the full model.
# The adapter checkpoint is tiny (a few MB) because it's just
# the small delta on top of the frozen base model weights.
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Adapter weights saved to: {OUTPUT_DIR}")
print("Files in output dir:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")

In [ ]:
# Optional: download the adapter weights from Colab to your local machine
import shutil
shutil.make_archive("/content/adapter_weights", "zip", OUTPUT_DIR)

from google.colab import files
files.download("/content/adapter_weights.zip")
print("Download started!")

## Step 12 — Quick inference test on val set

Before we run the full eval script, let's do a quick manual check
on a few val examples to make sure the model is generating valid JSON.

In [ ]:
# Run the model on a few val examples and print the results
model.eval()  # turn off dropout for inference

def run_inference(technician_note):
    """
    Takes a technician note as a string and returns the model's JSON output.
    """
    prompt = build_prompt(technician_note)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the model's response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,      # greedy decoding — deterministic and consistent
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the NEW tokens (skip the prompt tokens)
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return raw_output.strip()


# Test on first 3 val examples
print("=== Inference Test on Val Examples ===")
for i in range(3):
    note = val_data[i]["input"]
    expected = val_data[i]["output"]
    predicted_raw = run_inference(note)

    print(f"\n--- Example {i+1} ---")
    print(f"Input:     {note}")
    print(f"Expected:  {json.dumps(expected)}")
    print(f"Predicted: {predicted_raw}")

    # Try to parse the output as JSON
    try:
        parsed = json.loads(predicted_raw)
        print(f"JSON parse: OK")
    except json.JSONDecodeError:
        print(f"JSON parse: FAILED — model output is not valid JSON")

print("\nDone. Run eval.py for full evaluation.")